# CausalMan: causal inference data generation

This notebook generates the benchmark datasets for the CausalMan causal inference tasks.
For each (scale, seed) combination it produces:

| File | Description |
|---|---|
| `observational.csv.gz` | Training data sampled from the unmodified simulator |
| `task1_force_ltl_control/treatment.csv.gz` | Task 1 arms — intervention on `PF_M1_T1_Force_LTL` |
| `task2_force_control/treatment.csv.gz` | Task 2 arms — intervention on `PF_M1_T1_Force` |

Only observable nodes (as declared in the causal graph) are written to disk.
Each output file contains exactly `N_SAMPLES` rows.

**Edit the configuration cell below, then Run All.**

In [2]:
# ── The only cell you need to edit ────────────────────────────────────────────

SCALE       = "micro"   # "micro" | "small" | "medium" | "large"
SEEDS       = [42]      # full benchmark: [4, 6, 42, 66, 90]
N_SAMPLES   = 10_000    # rows to keep in each output CSV
OUTPUT_ROOT = "output/causalman_causal_inference"

# ─────────────────────────────────────────────────────────────────────────────

In [3]:
import os

import pandas as pd

from causalman import CausalMan

# Outcome variable — the binary process-result column we estimate effects on.
# Declared per scale so it can be updated if the column name differs across variants.
OUTCOME_BY_SCALE = {
    "micro":  "Sec_C2_Machine1_ProcessResult",
    "small":  "Sec_C2_Machine1_ProcessResult",
    "medium": "Sec_C2_Machine1_ProcessResult",
    "large":  "Sec_C2_Machine1_ProcessResult",
}

# Benchmark tasks — each task defines a pair of hard interventions.
# Task 1 raises the lower tolerance limit for the T1 press-fitting force.
# Task 2 raises the T1 press-fitting force itself to an abnormal level.
TASKS = [
    {
        "slug":      "task1_force_ltl",
        "variable":  "PF_M1_T1_Force_LTL",
        "control":   15000.0,
        "treatment": 18000.0,
    },
    {
        "slug":      "task2_force",
        "variable":  "PF_M1_T1_Force",
        "control":   16000.0,
        "treatment": 30000.0,
    },
]

In [ ]:
outcome = OUTCOME_BY_SCALE[SCALE]
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for seed in SEEDS:
    seed_dir = os.path.join(OUTPUT_ROOT, SCALE, f"seed_{seed:03d}")
    os.makedirs(seed_dir, exist_ok=True)
    print(f"\n── {SCALE}  seed={seed} ──")

    # ── Observational data ────────────────────────────────────────────────────
    sim = CausalMan(
        name=f"causalman_{SCALE}", seed=seed, batch_multiplier=1,
        parallelize=True, save_path=os.path.join(seed_dir, "observational"),
    )
    df_obs, _, _, dag = sim.sample()

    # Step 1: keep only nodes the DAG marks as observable.
    # The attribute may be stored as a Boolean or as the string "Observable".
    dag_observable = [
        n for n, d in dag.nodes(data=True)
        if d.get("Observable") in (True, "Observable") and n in df_obs.columns
    ]

    # Step 2: drop columns that are constant in the observational data.
    # Constant columns carry no information and are excluded by the observational notebook.
    observable = [col for col in dag_observable if df_obs[col].nunique(dropna=False) > 1]

    print(f"  observable columns : {len(observable)}  "
          f"({len(dag_observable) - len(observable)} constant columns removed)")

    if len(df_obs) < N_SAMPLES:
        raise RuntimeError(
            f"Simulator produced {len(df_obs)} rows but {N_SAMPLES} are required. "
            "Increase batch_multiplier or reduce N_SAMPLES."
        )

    df_obs[observable].sample(n=N_SAMPLES, random_state=seed).to_csv(
        os.path.join(seed_dir, "observational.csv"), index=False
    )
    print(f"  observational : {N_SAMPLES:,} rows saved")

    # ── Interventional arms ───────────────────────────────────────────────────
    # The observable list is fixed from the observational run above and reused
    # for all arms — ensuring consistent columns across the whole dataset.
    for task in TASKS:
        arms = {}
        for arm in ("control", "treatment"):
            sim = CausalMan(
                name=f"causalman_{SCALE}", seed=seed, batch_multiplier=1,
                parallelize=True, save_path=os.path.join(seed_dir, str(task["slug"]), arm),
            )
            sim.intervention_dict = {task["variable"]: task[arm]}
            df, _, _, _ = sim.sample()
            df[observable].sample(n=N_SAMPLES, random_state=seed).to_csv(
                os.path.join(seed_dir, f"{task['slug']}_{arm}.csv"),
                index=False,
            )
            arms[arm] = df[observable]

        # ATE = E[Y | do(treatment)] - E[Y | do(control)]
        ate = arms["treatment"][outcome].mean() - arms["control"][outcome].mean()
        print(f"  {task['slug']:<25} ATE = {ate:.4f}")

print(f"\nDone → {os.path.abspath(OUTPUT_ROOT)}")